<a href="https://colab.research.google.com/github/ajPRTS/aulasBase_analiseDeDados/blob/main/selenium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import re
import unicodedata
import requests
from bs4 import BeautifulSoup

# Transforma "Giruá" em "girua" e "São Paulo" em "sao-paulo".
def formatar_cidade(nome):
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "-", nome.lower()).strip("-")

cidade = input("Digite a cidade (ex.: Giruá): ").strip()
estado = input("Digite a sigla do estado (ex.: RS): ").strip().lower()

url = f"https://climabr.app/{estado}/{formatar_cidade(cidade)}/"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    # Faz a requisição HTTP direta para o site
    resposta = requests.get(url, headers=headers, timeout=20)

    if resposta.status_code == 404:
        print("\nCidade ou Estado não encontrados. Verifique a grafia e tente novamente.")
    elif resposta.status_code != 200:
        print(f"\nErro ao acessar o site. Código de resposta: {resposta.status_code}")
    else:
        # Processa o HTML usando BeautifulSoup
        sopa = BeautifulSoup(resposta.text, "html.parser")
        texto_pagina = sopa.get_text()

        # Expressão regular para encontrar as temperaturas
        padrao_temperaturas = (
            r"máxima\s+de\s+(-?\d+(?:[.,]\d+)?)\s*°\s*C"
            r"\s+e\s+mínima\s+de\s+(-?\d+(?:[.,]\d+)?)\s*°\s*C"
        )

        resultado = re.search(padrao_temperaturas, texto_pagina, flags=re.IGNORECASE)

        if resultado:
            maxima = float(resultado.group(1).replace(",", "."))
            minima = float(resultado.group(2).replace(",", "."))

            atualizacao = re.search(
                r"Atualizado em\s+(\d{2}/\d{2}/\d{4}(?:,\s*\d{2}:\d{2})?)",
                texto_pagina
            )

            print(f"\nCidade: {cidade.title()} / {estado.upper()}")
            print("Previsão identificada pelo site como hoje:")
            print(f"Temperatura mínima: {minima:.1f} °C")
            print(f"Temperatura máxima: {maxima:.1f} °C")

            if atualizacao:
                print(f"Atualização informada pelo site: {atualizacao.group(1)}")

            print(f"Fonte: {url}")
        else:
            print("\nNão foi possível extrair os dados de temperatura deste site. O layout da página pode ter mudado.")

except requests.exceptions.RequestException as erro:
    print(f"\nErro de conexão ao acessar o site: {erro}")

Digite a cidade (ex.: Giruá): Giruá
Digite a sigla do estado (ex.: RS): RS

Cidade: Giruá / RS
Previsão identificada pelo site como hoje:
Temperatura mínima: 8.0 °C
Temperatura máxima: 19.0 °C
Atualização informada pelo site: 19/10/2026, 12:14
Fonte: https://climabr.app/rs/girua/
